In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import lightning as L
from torch.utils.data import TensorDataset, DataLoader

In [2]:
class LSTMbyHand(L.LightningModule):
    def __init__(self):
        super().__init__()
        mean = torch.tensor(0.0)
        std = torch.tensor(1.0)

        # Forget gate
        self.wlr1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wlr2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.blr1 = nn.Parameter(torch.tensor(0.0), requires_grad=True)

        # Input gate
        self.wpr1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wpr2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bpr1 = nn.Parameter(torch.tensor(0.0), requires_grad=True)

        # Candidate cell state
        self.wp1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wp2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bp1 = nn.Parameter(torch.tensor(0.0), requires_grad=True)

        # Output gate
        self.wo1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wo2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bo1 = nn.Parameter(torch.tensor(0.0), requires_grad=True)

    def lstm_unit(self, input_value, long_memory, short_memory):
        # Forget Gate
        long_remember_percent = torch.sigmoid((short_memory * self.wlr1) + (input_value * self.wlr2) + self.blr1)

        # Input Gate
        potential_remember_percent = torch.sigmoid((short_memory * self.wpr1) + (input_value * self.wpr2) + self.bpr1)
        potential_memory = torch.tanh((short_memory * self.wp1) + (input_value * self.wp2) + self.bp1)

        updated_long_memory = ((long_memory * long_remember_percent) + (potential_remember_percent * potential_memory))

        # Output Gate
        output_percent = torch.sigmoid((short_memory * self.wo1) + (input_value * self.wo2) + self.bo1)
        updated_short_memory = torch.tanh(updated_long_memory) * output_percent

        return [updated_long_memory, updated_short_memory]

    def forward(self, input):
        long_memory = 0
        short_memory = 0
        day1 = input[0]
        day2 = input[1]
        day3 = input[2]
        day4 = input[3]

        long_memory, short_memory = self.lstm_unit(day1, long_memory, short_memory)
        long_memory, short_memory = self.lstm_unit(day2, long_memory, short_memory)
        long_memory, short_memory = self.lstm_unit(day3, long_memory, short_memory)
        long_memory, short_memory = self.lstm_unit(day4, long_memory, short_memory)

        return short_memory


    def configure_optimizers(self):
        return Adam(self.parameters())

    def training_step(self, batch, batch_idx):
        input_i , label_i = batch
        output_i = self.forward(input_i[0])
        loss = (output_i - label_i)**2

        self.log("train_loss", loss)

        if label_i == 0:
            self.log("out_0", output_i)
        else:
            self.log("out_1", output_i)

        return loss



In [3]:
model = LSTMbyHand()

print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =",
      model(torch.tensor([0.,0.5, 0.25, 1.])).detach())

print("Company A: Observed = 1, Predicted =",
      model(torch.tensor([1.,0.5, 0.25, 1.])).detach())



Now let's compare the observed and predicted values...
Company A: Observed = 0, Predicted = tensor(0.1065)
Company A: Observed = 1, Predicted = tensor(0.1149)


In [4]:
inputs = torch.tensor([[0.,0.5, 0.25, 1.], [1.,0.5, 0.25, 1.]])
labels = torch.tensor([0.,1.])

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

trainer = L.Trainer(max_epochs=2000)
trainer.fit(model, train_dataloaders=dataloader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name         | Type | Params | Mode | FLOPs
-----------------------------------------------------
  | other params | n/a  | 12     | n/a  | n/a  
-----------------------------------------------------
12        Trainable params
0         Non-trainable params
12        Total params
0.000     Total estimated model params size (MB)
0         Modules in train mode
0         Modules in eval mode
0         Total Flops
C:\Users\God -01.11\Desktop\AI Course\7-Neural Networks\NeuralNetworkF

Epoch 1999: 100%|██████████| 2/2 [00:00<00:00, 52.21it/s, v_num=9] 

`Trainer.fit` stopped: `max_epochs=2000` reached.


Epoch 1999: 100%|██████████| 2/2 [00:00<00:00, 41.65it/s, v_num=9]


In [5]:
print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =",
      model(torch.tensor([0.,0.5, 0.25, 1.])).detach())

print("Company A: Observed = 1, Predicted =",
      model(torch.tensor([1.,0.5, 0.25, 1.])).detach())


Now let's compare the observed and predicted values...
Company A: Observed = 0, Predicted = tensor(0.0005)
Company A: Observed = 1, Predicted = tensor(0.9289)


In [6]:
path_to_best_checkpoint = (
    "lightning_logs/version_7/checkpoints/epoch=1999-step=4000.ckpt"
)
trainer = L.Trainer(max_epochs=3000)
trainer.fit(model, train_dataloaders=dataloader, ckpt_path=path_to_best_checkpoint)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
Restoring states from the checkpoint path at lightning_logs/version_7/checkpoints/epoch=1999-step=4000.ckpt
C:\Users\God -01.11\Desktop\AI Course\7-Neural Networks\NeuralNetworkFromSkratch\.venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:566: The dirpath has changed from 'C:\\Users\\God -01.11\\Desktop\\AI Course\\7-Neural Networks\\NeuralNetworkFromSkratch\\lightning_logs\\version_7\\checkpoints' to 'C:\\Users\\God -01.11\\Desktop\\AI Course\\7-Neural Networks\\

Epoch 2999: 100%|██████████| 2/2 [00:00<00:00, 63.56it/s, v_num=10] 

`Trainer.fit` stopped: `max_epochs=3000` reached.


Epoch 2999: 100%|██████████| 2/2 [00:00<00:00, 39.87it/s, v_num=10]


In [7]:
print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =",
      model(torch.tensor([0.,0.5, 0.25, 1.])).detach())

print("Company A: Observed = 1, Predicted =",
      model(torch.tensor([1.,0.5, 0.25, 1.])).detach())


Now let's compare the observed and predicted values...
Company A: Observed = 0, Predicted = tensor(0.0042)
Company A: Observed = 1, Predicted = tensor(0.9312)


In [29]:
class LightningLSTM(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=1)

    def forward(self, input):
        input_trans = input.view(len(input), 1)
        lstm_out, temp = self.lstm(input_trans)

        prediction = lstm_out[-1]

        return prediction

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.01)

    def training_step(self, batch, batch_idx):
        input_i , label_i = batch
        output_i = self.forward(input_i[0])
        loss = (output_i - label_i)**2

        self.log("train_loss", loss)

        if label_i == 0:
            self.log("out_0", output_i)
        else:
            self.log("out_1", output_i)

        return loss

In [30]:
model = LightningLSTM()

print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =",
      model(torch.tensor([0.,0.5, 0.25, 1.])).detach())

print("Company A: Observed = 1, Predicted =",
      model(torch.tensor([1.,0.5, 0.25, 1.])).detach())


Now let's compare the observed and predicted values...
Company A: Observed = 0, Predicted = tensor([-0.2601])
Company A: Observed = 1, Predicted = tensor([-0.2583])


In [33]:
trainer = L.Trainer(max_epochs=3000, log_every_n_steps=2)
trainer.fit(model, train_dataloaders=dataloader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name | Type | Params | Mode  | FLOPs
----------------------------------------------
0 | lstm | LSTM | 16     | train | 0    
----------------------------------------------
16        Trainable params
0         Non-trainable params
16        Total params
0.000     Total estimated model params size (MB)
1         Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 2999: 100%|██████████| 2/2 [00:00<00:00, 107.56it/s, v_num=16]

`Trainer.fit` stopped: `max_epochs=3000` reached.


Epoch 2999: 100%|██████████| 2/2 [00:00<00:00, 76.18it/s, v_num=16] 


In [34]:
print("\nNow let's compare the observed and predicted values...")
print("Company A: Observed = 0, Predicted =",
      model(torch.tensor([0.,0.5, 0.25, 1.])).detach())

print("Company B: Observed = 1, Predicted =",
      model(torch.tensor([1.,0.5, 0.25, 1.])).detach())


Now let's compare the observed and predicted values...
Company A: Observed = 0, Predicted = tensor([0.0016])
Company B: Observed = 1, Predicted = tensor([0.9558])
